# test_future_order.py
Test achat/vente future EUR/USD CME -- paper trading, ordre MKT.

## Imports

In [1]:
from datetime import datetime
from ib_insync import IB, Contract, Order, util

util.patchAsyncio()

## Config

In [2]:
PORT      = 4002
CLIENT_ID = 15

ACTION = "BUY"       # "BUY" ou "SELL"
EXPIRY = "20260615"  # front future 6EM6
QTY    = 1

## Connect + Qualify Contract

In [3]:
ib = IB()
ib.connect("127.0.0.1", PORT, clientId=CLIENT_ID)
ib.reqMarketDataType(3)

# Qualifier le contrat
fut = Contract()
fut.symbol       = "EUR"
fut.secType      = "FUT"
fut.exchange     = "CME"
fut.currency     = "USD"
fut.lastTradeDateOrContractMonth = EXPIRY

details = ib.reqContractDetails(fut)
if not details:
    print("ERROR : contrat non trouve")
    ib.disconnect()
    raise SystemExit(1)

fut = details[0].contract
print(f"Contrat : {fut.localSymbol}  conId={fut.conId}")

Contrat : 6EM6  conId=496647057


Error 321, reqId 41: Error validating request.-'bP' : cause - Please enter exchange, contract: Future(conId=496647057, symbol='EUR', lastTradeDateOrContractMonth='20260615', multiplier='125000', currency='USD', localSymbol='6EM6', tradingClass='6E')
Error 300, reqId 41: Can't find EId with tickerId:41
Error 321, reqId 42: Error validating request.-'bP' : cause - Please enter exchange, contract: Future(conId=838628915, symbol='M6E', lastTradeDateOrContractMonth='20260615', multiplier='12500', currency='USD', localSymbol='M6EM6', tradingClass='M6E')


## Bid/Ask Snapshot

In [4]:
# Bid/ask avant ordre
ticker = ib.reqMktData(fut, "", False, False)
ib.sleep(4)

bid   = ticker.bid
ask   = ticker.ask
last  = ticker.last
close = ticker.close
mid   = round((bid + ask) / 2, 6) if bid and ask else None

print(f"bid={bid}  ask={ask}  last={last}  close={close}  mid={mid}")

entry_time  = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
entry_bid   = bid
entry_ask   = ask
entry_mid   = mid

ib.cancelMktData(fut)

bid=1.17195  ask=1.172  last=1.17195  close=1.1766  mid=1.171975


## Confirm + Send Order

In [5]:
# Confirmer
print(f"\nOrdre : {ACTION} {QTY} x {fut.localSymbol} MKT")
if input("Confirmer ? (y/n) : ").lower() != "y":
    print("Annule.")
    ib.disconnect()
    raise SystemExit(0)

# Envoyer
order = Order()
order.action        = ACTION
order.totalQuantity = QTY
order.orderType     = "MKT"
order.tif           = "DAY"

trade = ib.placeOrder(fut, order)
ib.sleep(3)

print(f"\nStatus   : {trade.orderStatus.status}")
print(f"Filled   : {trade.orderStatus.filled} / {QTY}")
print(f"AvgPrice : {trade.orderStatus.avgFillPrice}")


Ordre : BUY 1 x 6EM6 MKT

Status   : Filled
Filled   : 1.0 / 1
AvgPrice : 1.172


## Wait for Fill

In [6]:
# Attendre fill
for _ in range(15):
    ib.sleep(2)
    s = trade.orderStatus.status
    print(f"  [{s}]  filled={trade.orderStatus.filled}  avg={trade.orderStatus.avgFillPrice}")
    if s in ("Filled", "Cancelled", "Inactive"):
        break

fill_price = trade.orderStatus.avgFillPrice

  [Filled]  filled=1.0  avg=1.172


## Positions with Live Data

In [7]:
ib.sleep(1)
positions = ib.positions()

print(f"\n{'=' * 70}")
print("POSITIONS OUVERTES")
print(f"{'=' * 70}")

for p in positions:
    c = p.contract
    if c.secType != "FUT":
        continue

    tk = ib.reqMktData(c, "", False, False)
    ib.sleep(4)

    p_bid   = tk.bid
    p_ask   = tk.ask
    p_last  = tk.last
    p_close = tk.close
    p_mid   = round((p_bid + p_ask) / 2, 6) if p_bid and p_ask else None

    ib.cancelMktData(c)

    qty      = p.position
    avg_cost = p.avgCost
    multiplier = float(c.multiplier) if c.multiplier else 125000.0

    # P&L mid
    pnl_mid = round((p_mid - avg_cost) * qty * multiplier, 2) if p_mid else None

    print(f"\n  Symbole    : {c.localSymbol}")
    print(f"  Expiry     : {c.lastTradeDateOrContractMonth}")
    print(f"  Qty        : {qty:+.0f} contrat(s)")
    print(f"  Multiplier : {multiplier:,.0f} EUR/contrat")
    print(f"  -- Entree --")
    print(f"  Heure      : {entry_time}")
    print(f"  Bid/Ask    : {entry_bid} / {entry_ask}  mid={entry_mid}")
    print(f"  Fill price : {fill_price}")
    print(f"  -- Live --")
    print(f"  Bid/Ask    : {p_bid} / {p_ask}  mid={p_mid}")
    print(f"  Last       : {p_last}")
    print(f"  Close      : {p_close}")
    print(f"  -- P&L --")
    print(f"  AvgCost    : {avg_cost:.5f}")
    print(f"  P&L mid    : {pnl_mid:+.2f} USD" if pnl_mid else "  P&L mid    : --")
    notional = qty * multiplier * (p_mid if p_mid else avg_cost)
    print(f"  Notionnel  : {notional:,.0f} USD")

print(f"\n{'=' * 70}")
ib.disconnect()


POSITIONS OUVERTES

  Symbole    : 6EM6
  Expiry     : 20260615
  Qty        : -8 contrat(s)
  Multiplier : 125,000 EUR/contrat
  -- Entree --
  Heure      : 2026-04-13 12:11:02
  Bid/Ask    : 1.17195 / 1.172  mid=1.171975
  Fill price : 1.172
  -- Live --
  Bid/Ask    : nan / nan  mid=nan
  Last       : nan
  Close      : nan
  -- P&L --
  AvgCost    : 146503.15500
  P&L mid    : +nan USD
  Notionnel  : nan USD

  Symbole    : M6EM6
  Expiry     : 20260615
  Qty        : +11 contrat(s)
  Multiplier : 12,500 EUR/contrat
  -- Entree --
  Heure      : 2026-04-13 12:11:02
  Bid/Ask    : 1.17195 / 1.172  mid=1.171975
  Fill price : 1.172
  -- Live --
  Bid/Ask    : nan / nan  mid=nan
  Last       : nan
  Close      : nan
  -- P&L --
  AvgCost    : 14658.25091
  P&L mid    : +nan USD
  Notionnel  : nan USD



## Close Position

In [8]:
ib.connect("127.0.0.1", PORT, clientId=CLIENT_ID)
ib.reqMarketDataType(3)

# Ordre inverse pour fermer la position
CLOSE_ACTION = "SELL" if ACTION == "BUY" else "BUY"

print(f"Close : {CLOSE_ACTION} {QTY} x {fut.localSymbol} MKT")
if input("Confirmer close ? (y/n) : ").lower() != "y":
    print("Annule.")
    ib.disconnect()
    raise SystemExit(0)

close_order = Order()
close_order.action        = CLOSE_ACTION
close_order.totalQuantity = QTY
close_order.orderType     = "MKT"
close_order.tif           = "DAY"

close_trade = ib.placeOrder(fut, close_order)

for _ in range(15):
    ib.sleep(2)
    s = close_trade.orderStatus.status
    print(f"  [{s}]  filled={close_trade.orderStatus.filled}  avg={close_trade.orderStatus.avgFillPrice}")
    if s in ("Filled", "Cancelled", "Inactive"):
        break

close_price = close_trade.orderStatus.avgFillPrice
pnl_realized = round((close_price - fill_price) * QTY * 125000 * (1 if ACTION == "BUY" else -1), 2)

print(f"\nEntry  : {fill_price}")
print(f"Close  : {close_price}")
print(f"P&L    : {pnl_realized:+.2f} USD")

ib.disconnect()

Close : SELL 1 x 6EM6 MKT
  [Filled]  filled=1.0  avg=1.17195

Entry  : 1.172
Close  : 1.17195
P&L    : -6.25 USD


## Close All Positions

In [13]:
ib = IB()
ib.connect("127.0.0.1", PORT, clientId=CLIENT_ID)
ib.reqMarketDataType(3)

positions = ib.reqPositions()
ib.sleep(2)

open_pos = [p for p in positions if p.position != 0]
if not open_pos:
    print("No open positions.")
    ib.disconnect()
else:
    print(f"{'=' * 60}")
    print(f"{'Symbol':<15} {'Side':<6} {'Qty':>5}  {'Expiry':<10} {'AvgCost':>10}")
    print(f"{'=' * 60}")
    for p in open_pos:
        c = p.contract
        side = "BUY" if p.position > 0 else "SELL"
        print(f"{c.localSymbol:<15} {side:<6} {abs(p.position):>5.0f}  {c.lastTradeDateOrContractMonth:<10} {p.avgCost:>10.5f}")
    print(f"\nTotal: {len(open_pos)} position(s)")
    print("Closing all...\n")

    for p in open_pos:
        c = p.contract
        c.exchange = "CME"  # reqPositions() doesn't set exchange
        close_side = "SELL" if p.position > 0 else "BUY"
        qty = abs(p.position)

        order = Order()
        order.action = close_side
        order.totalQuantity = qty
        order.orderType = "MKT"
        order.tif = "DAY"

        trade = ib.placeOrder(c, order)
        print(f"  Closing {close_side} {qty:.0f} x {c.localSymbol}...")

        for _ in range(15):
            ib.sleep(2)
            s = trade.orderStatus.status
            print(f"    [{s}] filled={trade.orderStatus.filled} avg={trade.orderStatus.avgFillPrice}")
            if s in ("Filled", "Cancelled", "Inactive"):
                break

    # Verify
    ib.sleep(2)
    remaining = [p for p in ib.reqPositions() if p.position != 0]
    print(f"\n{'=' * 60}")
    print(f"Remaining positions: {len(remaining)}")
    for p in remaining:
        print(f"  {p.contract.localSymbol}  qty={p.position}")

    ib.disconnect()

Symbol          Side     Qty  Expiry        AvgCost
6EM6            SELL      11  20260615   146597.53000
M6EM6           BUY       11  20260615   14658.25091

Total: 2 position(s)
Closing all...

  Closing BUY 11 x 6EM6...
    [Filled] filled=11.0 avg=1.17335
  Closing SELL 11 x M6EM6...
    [Filled] filled=11.0 avg=1.1732

Remaining positions: 0
